# Silver Pre-EDA — Deliverable 1.2.1

This notebook completes the Silver Pre-EDA portion of the Medallion Architecture.

**Purpose:**  
To perform the initial cleaning, missing-value review, structural preparation, and readiness checks needed before deeper Silver-layer exploratory analysis.  
This includes validating the dataset structure, identifying early data quality issues, confirming sensor feature availability, and exporting a clean Silver-layer dataset.

**Outputs:**  
- Cleaned `pump__silver__train.parquet` ready for downstream EDA and modeling  
- Silver feature registry (`pump__silver__feature_registry.json`)  
- Status flags (`is_anomaly`, `is_normal`) and schema-aligned metadata  
- Structural consistency checks needed for the statistical comparison described in Section C of the project proposal

This deliverable ensures that the Silver layer provides a stable, reproducible foundation for both Silver EDA (Deliverable 1.2.2) and Gold modeling.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Pattern

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [2]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [3]:
def log_silver_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)

In [4]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [5]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [6]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "silver"
LAYER_NAME = "silver"
SILVER_VERSION = "silver__001"
CLEANING_RECIPE_ID = "silver__clean__dataset__agnostic__v001"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = None

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases

WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{SILVER_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Canonical Outputs

CANONICAL_OUTPUT_COLUMNS = [
    "event_time",
    "event_step",
    "time_index",
]

CANONICAL_NON_META_ORDER = [
    "event_time",
    "event_step",
    "time_index",
    "event_date",
]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Meta Columns

BRONZE_META_COLS = [
    "meta__dataset",
    "meta__split",
    "meta__run_id",
    "meta__asset_id",
    "meta__label_type",
    "meta__label_source",
    "meta__ingested_at_utc",
    "meta__source_file",
    "meta__source_row_id",
    "meta__record_id",
    "meta__schema_version"
]


SILVER_META_COLS_ADDED = [
    "meta__layer",
    "meta__processed_at_utc",
    "meta__silver_version",
    "meta__cleaning_recipe_id",
    "meta__dataset_name", 
    "meta__dataset_source",
    "meta__source_dataset",
    "meta__has_label_candidates",
    "meta__has_status_candidates",
    "meta__event_id",
    "meta__event_time_source",
    "meta__event_step_source",
    "meta__feature_set",
]



META_REQUIRED_COLUMNS = [
    "meta__dataset",
    "meta__split",
    "meta__run_id",
    "meta__asset_id",
    "meta__source_file",
    "meta__source_row_id",
]

META_SILVER_ADDED_COLUMNS = [
    "meta__layer",
    "meta__processed_at_utc",
    "meta__dataset_name",
    "meta__dataset_source",
    "meta__feature_set_id",
    "meta__feature_count",
    "meta__label_source_column",
    "meta__label_source_kind",
]


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Defaults / Fallbacks

# Fallbacks
ASSET_ID_DEFAULT_FALLBACK = "asset__001"
RUN_ID_DEFAULT_FALLBACK = "run__001"


# Defaults

RAW_PREFIX = "raw__"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


# Candidate lists

TIME_COLUMN_CANDIDATES=["timestamp", "event_time", "time", "datetime", "date"]
STEP_COLUMN_CANDIDATES=["cycle", "step", "event_step", "sample", "time_index"]

TIE_BREAKER_CANDIDATES = ["meta__source_row_id", "meta__record_id"]

STATUS_COLUMN_CANDIDATES = ["machine_status", "status", "state", "condition"]
LABEL_COLUMN_CANDIDATES = ["anomaly_flag", "fault_flag", "target", "label", "y"]

#BRONZE_DATASET_SOURCE_COLUMN = "meta__datset"
DATASET_CANDIDATES = ["meta__dataset_name", "meta__source_dataset"]

#### #### #### #### #### #### #### #### 

NORMAL_STATUS_VALUES = {"normal", "ok", "healthy", "running"} 

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Regex Matching

RegexLike = Union[str, Pattern[str]]

# Unnamed Column Names
UNNAMED_COLUMN_REGEX  = re.compile(r"^unnamed:\s*\d+(\.\d+)?$", flags=re.IGNORECASE)


#### #### #### #### #### #### #### #### 

JUNK_COLUMN_CANDIDATES = {"Unnamed:", "Unnamed", "unnamed", "...", "level_0", "", " ", "\t", "\ufeff", "\ufeff<name>"}


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# QA/EDA thresholds

# Thresholds
MIN_TIME_PARSE_SUCCESS_PERCENT = 95.0
MIN_STEP_PARSE_SUCCESS_PERCENT = 95.0

QUARANTINE_MISSING_PCT = 30.0
CORRELATION_THRESHOLD = 0.85

# EDA Controls
TOP_N_SENSORS_FOR_PLOTS = 8
PAIRPLOT_SENSOR_CAP = 8
PAIRPLOT_SAMPLE_N = 2000
TOP_PLOT_COLS = 8
TOP_CORR_COLS = 15

ROLLING_MINUTES = 60
LOOKBACK_HOURS = 48
BASELINE_DAYS = 7
BASELINE_GAP_HOURS = 12
SUSTAIN_MINUTES = 180
TOP_SENSOR_PRE_HOURS = 6


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File Names
BRONZE_TRAIN_DATA_FILE_NAME = "pump__bronze__train.parquet"
SILVER_TRAIN_DATA_FILE_NAME = "pump__silver__train.parquet"


#TIME_EVENT_COLUMN = None

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CANONICAL_EXCLUDE_COLUMNS = [
    "event_time",
    "event_step",
    "time_index",
    "event_date",
    "meta__event_id",
    "meta__event_time_source",
    "meta__event_step_source",
    "meta__event_time_parse_success_percent",
    "meta__event_step_parse_success_percent",
]

LABEL_EXCLUDE_COLUMNS = [
    "anomaly_flag",
    "is_anomaly",
    "is_normal",
    "status_normal_value",
    "meta__label_source",
    "meta__label_type",
]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


CANONICAL_NON_META_ORDER = ["event_time", "event_step", "time_index", "event_date"]

LABEL_COLUMNS_ORDER = [
    "anomaly_flag",
    "is_anomaly",
    "is_normal",
    "status_normal_value",
#    "meta__label_source",
#    "meta__label_type",
]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####





In [7]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

In [8]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Paths
BRONZE_TRAIN_DATA_PATH = paths.data_bronze_train
SILVER_TRAIN_DATA_PATH = paths.data_silver_train

# Will create dataset specific after resolving dataset name
SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" 

LOGS_PATH = paths.logs

# Path Failsafes
SILVER_TRAIN_DATA_PATH.mkdir(parents=True, exist_ok=True)
SILVER_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)


In [9]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [10]:
# Logging Setup

# Create silver log path 
silver_log_path = paths.logs / "silver.log"

# Initial Logger
configure_logging(
    "capstone",
    silver_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.silver")

# Log load and initiation
logger.info("Silver stage starting")

# Log paths loads
log_silver_paths(paths, logger)


2026-03-01 04:18:12,307 | INFO | capstone.silver | Silver stage starting
2026-03-01 04:18:12,310 | INFO | capstone.silver | Project Root Path Loaded: /workspace
2026-03-01 04:18:12,312 | INFO | capstone.silver | Project Logging Path Loaded: /workspace/logs
2026-03-01 04:18:12,319 | INFO | capstone.silver | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-01 04:18:12,322 | INFO | capstone.silver | Project Notebooks Path Loaded /workspace/notebooks
2026-03-01 04:18:12,324 | INFO | capstone.silver | Project Data Path Loaded: /workspace/data
2026-03-01 04:18:12,326 | INFO | capstone.silver | Data Bronze Path Loaded: /workspace/data/bronze
2026-03-01 04:18:12,327 | INFO | capstone.silver | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-03-01 04:18:12,328 | INFO | capstone.silver | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-03-01 04:18:12,329 | INFO | capstone.silver | Data Silver Path Loaded: /workspace/data/silver
2026-03-01 04:18:12,3

In [11]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [12]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="silver",
    config={
        "silver_version": SILVER_VERSION,
        "cleaning_recipe_id": CLEANING_RECIPE_ID,
        "quarantine_missing_pct": QUARANTINE_MISSING_PCT,
        "min_time_parse_success_percent": MIN_TIME_PARSE_SUCCESS_PERCENT,
        "rolling_window": ROLLING_MINUTES,
        "bronze_path": str(BRONZE_TRAIN_DATA_PATH / BRONZE_TRAIN_DATA_FILE_NAME),
        "silver_out_dir": str(SILVER_TRAIN_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-03-01 04:18:18,081 | INFO | capstone.silver | W&B initialized: silver__001


In [13]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [14]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=CLEANING_RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-01 04:18:18,718 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:18.718540+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-01T04:18:18.718540+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

In [15]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

### Load Data

In [16]:
# Load Data
# dataframe = load_data(paths.data_bronze, "pump__bronze__train.parquet")

preferred_bronze = BRONZE_TRAIN_DATA_PATH / BRONZE_TRAIN_DATA_FILE_NAME

if preferred_bronze.exists():
    bronze_data_path = preferred_bronze
else:
    parquet_files = sorted(BRONZE_TRAIN_DATA_PATH.glob("*.parquet"))
    if len(parquet_files) == 0:
        raise FileNotFoundError(f"No parquet files found in {BRONZE_TRAIN_DATA_PATH}")
    if len(parquet_files) > 1: 
        logger.warning("Multiple Parquet Files found; Using First %s", parquet_files[0])
    bronze_data_path = parquet_files[0]

if not bronze_data_path.exists():
    raise FileNotFoundError(f"Bronze parquet not found: {bronze_data_path}")
    
dataframe = load_data(bronze_data_path.parent, bronze_data_path.name)



#### #### #### #### #### #### #### #### 

logger.info("Loaded Bronze: %s | shape=%s", bronze_data_path, dataframe.shape)
wandb_run.log({"bronze_rows": int(dataframe.shape[0]), "bronze_cols": int(dataframe.shape[1])})

ledger.add(
    kind="step",
    step="load_bronze",
    message="Loaded Bronze Parquet",
    why="Silver must be derived from reprodicible Bronze Artifact",
    consequence="All silver outputs trace back to this file",
    data={"bronze_path": str(bronze_data_path), "shape": list(dataframe.shape), "cols": len(dataframe.columns)},
    logger=logger
)


#### #### #### #### #### #### #### #### 

display(dataframe.head(3))

2026-03-01 04:18:19,389 | INFO | capstone.file_io | Loading Parquet: /workspace/data/bronze/train/pump__bronze__train.parquet


2026-03-01 04:18:20,033 | INFO | capstone.silver | Loaded Bronze: /workspace/data/bronze/train/pump__bronze__train.parquet | shape=(220320, 65)
2026-03-01 04:18:20,036 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:20.036050+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'load_bronze', 'message': 'Loaded Bronze Parquet', 'why': 'Silver must be derived from reprodicible Bronze Artifact', 'consequence': 'All silver outputs trace back to this file', 'data': {'bronze_path': '/workspace/data/bronze/train/pump__bronze__train.parquet', 'shape': [220320, 65], 'cols': 65}}


,meta__dataset,meta__split,meta__run_id,meta__asset_id,meta__label_type,meta__label_source,meta__ingested_at_utc,meta__source_file,meta__source_row_id,meta__record_id,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump,unsplit,run__001,asset__001,<NA>,<NA>,2026-03-01 03:54:51.311816+00:00,sensor.csv,0,14598431322315673869,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,pump,unsplit,run__001,asset__001,<NA>,<NA>,2026-03-01 03:54:51.311816+00:00,sensor.csv,1,15954729095895098000,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,pump,unsplit,run__001,asset__001,<NA>,<NA>,2026-03-01 03:54:51.311816+00:00,sensor.csv,2,10041703297090838359,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.39757,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,NaN,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL


In [17]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [18]:
print("=== Silver Pre-EDA: Data Overview ===")
print(f"Shape: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns\n")

print("=== Column Types ===")
display(dataframe.dtypes.to_frame("dtype").head(20))

print("\n=== df.info() ===")
dataframe.info()

print("\n=== Basic Descriptive Statistics (numeric only) ===")
display(dataframe.describe().T)

=== Silver Pre-EDA: Data Overview ===
Shape: 220,320 rows × 65 columns

=== Column Types ===


,dtype
meta__dataset,category
meta__split,category
meta__run_id,object
meta__asset_id,object
meta__label_type,string[python]
meta__label_source,string[python]
meta__ingested_at_utc,"datetime64[us, UTC]"
meta__source_file,string[python]
meta__source_row_id,int64
meta__record_id,uint64



=== df.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 65 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   meta__dataset          220320 non-null  category           
 1   meta__split            220320 non-null  category           
 2   meta__run_id           220320 non-null  object             
 3   meta__asset_id         220320 non-null  object             
 4   meta__label_type       0 non-null       string             
 5   meta__label_source     0 non-null       string             
 6   meta__ingested_at_utc  220320 non-null  datetime64[us, UTC]
 7   meta__source_file      220320 non-null  string             
 8   meta__source_row_id    220320 non-null  int64              
 9   meta__record_id        220320 non-null  uint64             
 10  Unnamed: 0             220320 non-null  int64              
 11  timestamp           

,count,mean,std,min,25%,50%,75%,max
meta__source_row_id,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
meta__record_id,220320.0,9.224243e+18,5.322367e+18,1.873763e+14,4.610703e+18,9.220552e+18,1.381850e+19,1.844674e+19
Unnamed: 0,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
sensor_00,210112.0,2.372221e+00,4.122274e-01,0.000000e+00,2.438831e+00,2.456539e+00,2.499826e+00,2.549016e+00
sensor_01,219951.0,4.759161e+01,3.296666e+00,0.000000e+00,4.631076e+01,4.813368e+01,4.947916e+01,5.672743e+01
sensor_02,220301.0,5.086739e+01,3.666820e+00,3.315972e+01,5.039062e+01,5.164930e+01,5.277777e+01,5.603299e+01
sensor_03,220301.0,4.375248e+01,2.418887e+00,3.164062e+01,4.283854e+01,4.422743e+01,4.531250e+01,4.822049e+01
sensor_04,220301.0,5.906739e+02,1.440239e+02,2.798032e+00,6.266204e+02,6.326389e+02,6.376157e+02,8.000000e+02
sensor_05,220301.0,7.339641e+01,1.729825e+01,0.000000e+00,6.997626e+01,7.557679e+01,8.091215e+01,9.999988e+01
sensor_06,215522.0,1.350154e+01,2.163736e+00,1.446759e-02,1.334635e+01,1.364294e+01,1.453993e+01,2.225116e+01


In [19]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [20]:
def remove_junk_import_columns(
        dataframe: pd.DataFrame, 
        *,
        junk_column_candidates: List[str],
        regex_pattern: Optional[RegexLike] = None,
        default_pattern: RegexLike = UNNAMED_COLUMN_REGEX,
    ) -> Tuple[pd.DataFrame, List[str], str]:

    dataframe_copy = dataframe.copy()
    
    def _ensure_regex_pattern(
            regex: Optional[RegexLike],
            *,
            default: RegexLike,
            flags: int = re.IGNORECASE,
        ) -> Pattern[str]:

        if regex is None:
            regex = default
        if isinstance(regex, re.Pattern):
            return regex
        if isinstance(regex, str):
            return re.compile(regex, flags=flags)

        raise TypeError(f"Regex must be a str, re.Pattern, or None; Instead got {type(regex)}")

    
    pattern = _ensure_regex_pattern(regex_pattern, default=default_pattern, flags=re.IGNORECASE)

    normalize_candidates = [
        str(candidate).strip().lower()
        for candidate in junk_column_candidates
        if candidate is not None and str(candidate).strip() != ""
        ]

    junk_prefixes = tuple(normalize_candidates)

    junk_columns_found: list[str] = []

    for column in dataframe_copy.columns:
        column_string = str(column)
        column_normalized = column_string.strip().lower()

        is_junk_prefix = column_normalized.startswith(junk_prefixes) if junk_prefixes else False
        is_regex_match = bool(pattern.search(column_string))

        if is_junk_prefix or is_regex_match:
            junk_columns_found.append(column_string)

    if junk_columns_found:
        dataframe_copy = dataframe_copy.drop(columns=junk_columns_found, errors="ignore")

    return dataframe_copy, junk_columns_found, pattern.pattern


In [21]:
dataframe, junk_columns_found, pattern_used = remove_junk_import_columns(
    dataframe,
    junk_column_candidates=JUNK_COLUMN_CANDIDATES,
    regex_pattern=UNNAMED_COLUMN_REGEX,
    default_pattern=UNNAMED_COLUMN_REGEX,
)

#### #### #### #### #### #### #### #### 


if junk_columns_found:

    logger.info("Dropped junk columns: %s", junk_columns_found)

    ledger.add(
        kind="step",
        step="remove_junk_import_columns",
        message="Dropped junk columns via candidates and/or regex pattern",
        data={"dropped": junk_columns_found, "pattern": pattern_used},
        logger=logger,
    )
else:
    logger.info("No junk columns found.")

    ledger.add(
        kind="step",
        step="remove_junk_import_columns",
        message="No junk columns matched candidates/regex pattern",
        data={"dropped": [], "pattern": pattern_used},
        logger=logger,
    )




        

#### #### #### #### #### #### #### #### 

2026-03-01 04:18:22,549 | INFO | capstone.silver | Dropped junk columns: ['Unnamed: 0']
2026-03-01 04:18:22,551 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:22.551843+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'remove_junk_import_columns', 'message': 'Dropped junk columns via candidates and/or regex pattern', 'why': None, 'consequence': None, 'data': {'dropped': ['Unnamed: 0'], 'pattern': '^unnamed:\\s*\\d+(\\.\\d+)?$'}}


In [22]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [23]:
def deduplicate_columns(
    dataframe: pd.DataFrame,
    *,
    keep: str = "first",
) -> Tuple[pd.DataFrame, List[str]]:
    
    dataframe = dataframe.copy()
    
    if dataframe.columns.is_unique:
        return dataframe, []
    
    duplicates_found = dataframe.columns[dataframe.columns.duplicated()].tolist()
    dataframe = dataframe.loc[:, ~dataframe.columns.duplicated(keep=keep)]
    
    return dataframe, duplicates_found




In [24]:
dataframe, duplicates_columns_found = deduplicate_columns(dataframe, keep="first")

if duplicates_columns_found:
    logger.warning("Duplicate column names removed (kept first): %s", duplicates_columns_found)
else:
    logger.info("No duplicate column names detected.")

ledger.add(
    kind="step",
    step="deduplicate_columns",
    message="Removed duplicate column names (kept first occurrence).",
    data={"duplicates": duplicates_columns_found, "count": len(duplicates_columns_found)},
    logger=logger,
)

2026-03-01 04:18:23,462 | INFO | capstone.silver | No duplicate column names detected.
2026-03-01 04:18:23,465 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:23.465003+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'deduplicate_columns', 'message': 'Removed duplicate column names (kept first occurrence).', 'why': None, 'consequence': None, 'data': {'duplicates': [], 'count': 0}}


{'ts_utc': '2026-03-01T04:18:23.465003+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'deduplicate_columns',
 'message': 'Removed duplicate column names (kept first occurrence).',
 'why': None,
 'consequence': None,
 'data': {'duplicates': [], 'count': 0}}

### Add Silver Meta Columns

In [25]:
dataframe = dataframe.copy()


# Add Layer Tag
dataframe["meta__layer"] = LAYER_NAME

# Get Current Timestamp
processed_at_utc = pd.Timestamp.now(tz="UTC")

# Add Silver Processing Column and timestamp start
dataframe["meta__processed_at_utc"] = processed_at_utc

# Add Silver Version Meta Column
dataframe["meta__silver_version"] = SILVER_VERSION

# Add Silver Cleaning Recipe ID Meta Column
if "meta__cleaning_recipe_id" not in dataframe.columns:
    dataframe["meta__cleaning_recipe_id"] = pd.NA
dataframe["meta__cleaning_recipe_id"] = CLEANING_RECIPE_ID


# Ensure Bronze-required meta columns exist (only fill if missing)
for column_name in META_REQUIRED_COLUMNS:
    if column_name not in dataframe.columns:
        # Choose a safe default by column
        if column_name == "meta__dataset":
            dataframe[column_name] = pd.NA
        elif column_name == "meta__split":
            dataframe[column_name] = "unsplit"
        elif column_name == "meta__run_id":
            dataframe[column_name] = "run__single"
        elif column_name == "meta__asset_id":
            dataframe[column_name] = "asset__single"
        elif column_name == "meta__source_file":
            dataframe[column_name] = ""
        elif column_name == "meta__source_row_id":
            dataframe[column_name] = pd.RangeIndex(start=0, stop=len(dataframe), step=1)
        else:
            dataframe[column_name] = pd.NA



#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="silver_meta_contract",
    message="Stamped Silver-owned meta columns and ensured required Bronze meta columns exist.",
    why="Silver needs consistent lineage + processing timestamps before canonical ordering, labels, and features.",
    consequence="Downstream steps can rely on meta__ fields without overwriting Bronze meaning.",
    data={
        "layer": "silver",
        "processed_at_utc": processed_at_utc.isoformat(),
        "required_meta_columns": list(META_REQUIRED_COLUMNS),
        "added_meta_columns": list(META_SILVER_ADDED_COLUMNS),
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

2026-03-01 04:18:23,787 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:23.787427+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'silver_meta_contract', 'message': 'Stamped Silver-owned meta columns and ensured required Bronze meta columns exist.', 'why': 'Silver needs consistent lineage + processing timestamps before canonical ordering, labels, and features.', 'consequence': 'Downstream steps can rely on meta__ fields without overwriting Bronze meaning.', 'data': {'layer': 'silver', 'processed_at_utc': '2026-03-01T04:18:23.781532+00:00', 'required_meta_columns': ['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__source_file', 'meta__source_row_id'], 'added_meta_columns': ['meta__layer', 'meta__processed_at_utc', 'meta__dataset_name', 'meta__dataset_source', 'meta__feature_set_id', 'meta__feature_count', 'meta__label_source_column', 'meta__label_source_kind'], 'shape': {'rows': 220320, 'cols

{'ts_utc': '2026-03-01T04:18:23.787427+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_meta_contract',
 'message': 'Stamped Silver-owned meta columns and ensured required Bronze meta columns exist.',
 'why': 'Silver needs consistent lineage + processing timestamps before canonical ordering, labels, and features.',
 'consequence': 'Downstream steps can rely on meta__ fields without overwriting Bronze meaning.',
 'data': {'layer': 'silver',
  'processed_at_utc': '2026-03-01T04:18:23.781532+00:00',
  'required_meta_columns': ['meta__dataset',
   'meta__split',
   'meta__run_id',
   'meta__asset_id',
   'meta__source_file',
   'meta__source_row_id'],
  'added_meta_columns': ['meta__layer',
   'meta__processed_at_utc',
   'meta__dataset_name',
   'meta__dataset_source',
   'meta__feature_set_id',
   'meta__feature_count',
   'meta__label_source_column',
   'meta__label_source_kind'],
  'shape': {'rows': 220320, 'cols': 68}}}

In [26]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

### Capture Dataset Name

In [27]:
def resolve_dataset_name(
        dataframe: pd.DataFrame, 
        *, 
        dataset_candidates: List[str],
        config_value: Optional[str] = None,
        fallback_value: Optional[str] = None,
        bronze_source_column: str = "meta__dataset",
        fail_on_multiple_in_bronze: bool = True,
    ) -> Tuple[str, str, str]:
    

    def _clean_values(series: pd.Series) -> pd.Series:
        values = (
            series.dropna()
            .astype("string")
            .str.strip()
        )
        # Ingores empty strings check
        return values[values != ""]


    # Bronze Source Column
    if bronze_source_column in dataframe.columns:
        values = _clean_values(dataframe[bronze_source_column])

        if len(values) > 0:
            unique_values = values.unique()

            if len(unique_values) == 1:
                return str(unique_values[0]), bronze_source_column, "unique"
        
            if fail_on_multiple_in_bronze:
                raise ValueError(
                    f"Multiple values are not allowed from bronze source '{bronze_source_column}'. "
                    f"Values discovered: {unique_values[:10]}"
                )

            top_value = values.value_counts().index[0]
            
            return str(top_value), bronze_source_column, "mode"
    
    # Dataset Column Candidate
    for column in dataset_candidates:
        if column == bronze_source_column:
            continue
        if column not in dataframe.columns:
            continue

        values = _clean_values(dataframe[column])
        
        if len(values) == 0:
            continue 
    
        unique_values = values.unique()
            
        if len(unique_values) == 1:
            return str(unique_values[0]), column, "unique"
        
        top_value = values.value_counts().index[0]
        return str(top_value), column, "mode"
    

    # Config Source
    if config_value is not None and str(config_value).strip() != "":
        return str(config_value).strip(), "config", "config"

    # Fallback Source
    fallback_value_text = (
        fallback_value 
        if (fallback_value is not None and str(fallback_value).strip() !="") 
        else "unknown_dataset"
    )
    return str(fallback_value_text).strip(), "fallback", "fallback"




In [28]:

DATASET_NAME, DATASET_SOURCE_COLUMN, DATASET_METHOD = resolve_dataset_name(dataframe, 
                                                                           dataset_candidates=DATASET_CANDIDATES, 
                                                                           config_value=DATASET_NAME_CONFIG, 
                                                                           fallback_value="unknown_dataset", 
                                                                           fail_on_multiple_in_bronze=False
                                                                           )

DATASET_NAME = str(DATASET_NAME).strip().lower()

dataframe["meta__dataset_name"] = DATASET_NAME
dataframe["meta__dataset_source"] = DATASET_SOURCE_COLUMN

# Only backfill meta__dataset if it's missing or empty - We do not want to blindly overwrite bronze
if "meta__dataset" not in dataframe.columns:
    dataframe["meta__dataset"] = DATASET_NAME
else:
    empty_mask = dataframe["meta__dataset"].isna() | (dataframe["meta__dataset"].astype("string").str.strip() == "")
    if empty_mask.any():
        dataframe.loc[empty_mask, "meta__dataset"] = DATASET_NAME


#### #### #### #### #### #### #### #### 

ledger.add(
    kind="decision",
    step="resolve_dataset_name",
    message="Resolved dataset name for Silver (Bronze preferred; fallback otherwise).",
    why="Silver uses Bronze-stamped dataset identifier as source of truth to preserve lineage.",
    consequence="All artifacts and feature_set IDs will be consistent across layers.",
    data={
        "dataset_name": DATASET_NAME,
        "dataset_source_col": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
        "candidates": DATASET_CANDIDATES,
        
        },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

2026-03-01 04:18:24,721 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:24.721435+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'resolve_dataset_name', 'message': 'Resolved dataset name for Silver (Bronze preferred; fallback otherwise).', 'why': 'Silver uses Bronze-stamped dataset identifier as source of truth to preserve lineage.', 'consequence': 'All artifacts and feature_set IDs will be consistent across layers.', 'data': {'dataset_name': 'pump', 'dataset_source_col': 'meta__dataset', 'dataset_method': 'unique', 'candidates': ['meta__dataset_name', 'meta__source_dataset']}}


{'ts_utc': '2026-03-01T04:18:24.721435+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'resolve_dataset_name',
 'message': 'Resolved dataset name for Silver (Bronze preferred; fallback otherwise).',
 'why': 'Silver uses Bronze-stamped dataset identifier as source of truth to preserve lineage.',
 'consequence': 'All artifacts and feature_set IDs will be consistent across layers.',
 'data': {'dataset_name': 'pump',
  'dataset_source_col': 'meta__dataset',
  'dataset_method': 'unique',
  'candidates': ['meta__dataset_name', 'meta__source_dataset']}}

In [29]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [30]:
# Creating Dataset Folder inside Aritifacts Silver 

SILVER_ARTIFACTS_PATH = SILVER_ARTIFACTS_PATH / DATASET_NAME
SILVER_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Silver pre-EDA artifacts folder: {SILVER_ARTIFACTS_PATH}")

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="decision",
    step="create dataset specific artifcats folder",
    message="Creating a dataset specific artificats folder now that the dataset name has been resolved",
    why="To allow for artifacts to be stored properly. ",
    consequence="All artifacts will intermingle and create the need for organization/decluttering. .",
    data={
        "dataset_name": DATASET_NAME,
        "silver_artifacts_folder": SILVER_ARTIFACTS_PATH
        },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

2026-03-01 04:18:25,350 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:25.350613+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'create dataset specific artifcats folder', 'message': 'Creating a dataset specific artificats folder now that the dataset name has been resolved', 'why': 'To allow for artifacts to be stored properly. ', 'consequence': 'All artifacts will intermingle and create the need for organization/decluttering. .', 'data': {'dataset_name': 'pump', 'silver_artifacts_folder': PosixPath('/workspace/artifacts/silver/pump')}}


Silver pre-EDA artifacts folder: /workspace/artifacts/silver/pump


{'ts_utc': '2026-03-01T04:18:25.350613+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'create dataset specific artifcats folder',
 'message': 'Creating a dataset specific artificats folder now that the dataset name has been resolved',
 'why': 'To allow for artifacts to be stored properly. ',
 'consequence': 'All artifacts will intermingle and create the need for organization/decluttering. .',
 'data': {'dataset_name': 'pump',
  'silver_artifacts_folder': PosixPath('/workspace/artifacts/silver/pump')}}

In [31]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [32]:
def resolve_label_or_status_source(
    dataframe: pd.DataFrame, 
    *,
    label_candidates: list[str], 
    status_candidates: list[str],
    top_n: int = 10,
) -> tuple[Optional[str], str, Dict[str, Any]]:
    
    # Helper function to get value counts from dataframe column
    # holds the top n amount and convert the the value count information into 
    # keys to string format so ledger serialization works well
    def _top_values(column: str) -> Dict[str, int]:
        value_counts = dataframe[column].value_counts(dropna=False).head(top_n)
        return {str(key): int(value) for key, value in value_counts.items()}


    def _column_info(column: str) -> Dict[str, Any]:
        total_rows = int(len(dataframe))
        non_null = int(dataframe[column].notna().sum())
        unique = int(dataframe[column].nunique(dropna=False))
        return {
            "top_values": _top_values(column),
            "unique_count": unique,
            "non_null_count": non_null,
            "total_row_count": total_rows,
            "coverage_pct": float((non_null / total_rows) * 100.0) if total_rows > 0 else 0.0,
        }

    found_labels: Dict[str, Any] = {}
    found_status: Dict[str, Any] = {}

    
    # Label Candidates
    for column in label_candidates:
        if column in dataframe.columns:
            found_labels[column] = _column_info(column)
        
    # Status Candidates
    for column in status_candidates:
        if column in dataframe.columns:
            found_status[column] = _column_info(column)


    # Choose primary source (policy: label preferred)
    if len(found_labels) > 0:
        chosen_column = next(iter(found_labels.keys()))
        chosen_type = "label"
        chosen_info = found_labels[chosen_column]
        chosen_from = "label_candidates"
    elif len(found_status) > 0:
        chosen_column = next(iter(found_status.keys()))
        chosen_type = "status"
        chosen_info = found_status[chosen_column]
        chosen_from = "status_candidates"
    else:
        return None, "none", {
            "chosen_from": "none",
            "found_labels": {},
            "found_status": {},
            "top_values": {},
            "unique_count": 0,
            "non_null_count": 0,
            "total_row_count": int(len(dataframe)),
            "coverage_pct": 0.0,
        }

    # Pack full context (so we know if both existed)
    info = {
        "chosen_from": chosen_from,
        "found_labels": found_labels,
        "found_status": found_status, 
        **chosen_info,                  
    }

    return chosen_column, chosen_type, info

In [33]:
dataframe.columns

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__label_type', 'meta__label_source', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id',
       'meta__record_id', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25',
       'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39',
       'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status', 'meta__layer',
       'meta__processed_at_utc', 'meta__silver_

In [34]:
LABEL_SOURCE_COLUMN, LABEL_SOURCE_TYPE, LABEL_SOURCE_INFO = resolve_label_or_status_source(dataframe, 
                                                                                           label_candidates = LABEL_COLUMN_CANDIDATES, 
                                                                                           status_candidates = STATUS_COLUMN_CANDIDATES, 
                                                                                           top_n = 10)

dataframe["meta__label_source_column"] = LABEL_SOURCE_COLUMN if LABEL_SOURCE_COLUMN else ""
dataframe["meta__label_source_kind"] = LABEL_SOURCE_TYPE



#### #### #### #### #### #### #### #### 

# Build small summaries
found_label_columns = list(LABEL_SOURCE_INFO.get("found_labels", {}).keys())
found_status_columns = list(LABEL_SOURCE_INFO.get("found_status", {}).keys())

# Meta stamp the candidates for tracking. 
dataframe["meta__has_label_candidates"] = int(len(found_label_columns) > 0)
dataframe["meta__has_status_candidates"] = int(len(found_status_columns) > 0)


chosen_summary = {
    "top_values": LABEL_SOURCE_INFO.get("top_values", {}),
    "unique_count": LABEL_SOURCE_INFO.get("unique_count", 0),
    "non_null_count": LABEL_SOURCE_INFO.get("non_null_count", 0),
    "total_row_count": LABEL_SOURCE_INFO.get("total_row_count", int(len(dataframe))),
    "coverage_pct": LABEL_SOURCE_INFO.get("coverage_pct", 0.0),
}

#### #### #### #### 

ledger.add(
    kind="decision",
    step="resolve_label_or_status_source",
    message="Resolved label/status source column for Silver.",
    why="Silver needs one consistent source to build anomaly_flag and state flags.",
    consequence="Downstream label building uses this source consistently. we also record all detected candidates for traceability.",
    data={
        # Primary decision
        "label_source_col": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        "chosen_from": LABEL_SOURCE_INFO.get("chosen_from", "none"),

        # Candidate inputs
        "label_candidates": list(LABEL_COLUMN_CANDIDATES),
        "status_candidates": list(STATUS_COLUMN_CANDIDATES),

        # What we found - Summary
        "found_label_columns": found_label_columns,
        "found_status_columns": found_status_columns,
        "found_counts": {
            "labels_found": int(len(found_label_columns)),
            "status_found": int(len(found_status_columns)),
        },

        # Chosen column stats (small and high-signal)
        "chosen_column_stats": chosen_summary,

        # Optional: full details (Commented out to reduce ledger size)
        #"all_matches_detail": {
        #    "found_labels": LABEL_SOURCE_INFO.get("found_labels", {}),
        #    "found_status": LABEL_SOURCE_INFO.get("found_status", {}),
        #},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 




2026-03-01 04:18:26,515 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:26.515925+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'resolve_label_or_status_source', 'message': 'Resolved label/status source column for Silver.', 'why': 'Silver needs one consistent source to build anomaly_flag and state flags.', 'consequence': 'Downstream label building uses this source consistently. we also record all detected candidates for traceability.', 'data': {'label_source_col': 'machine_status', 'label_source_type': 'status', 'chosen_from': 'status_candidates', 'label_candidates': ['anomaly_flag', 'fault_flag', 'target', 'label', 'y'], 'status_candidates': ['machine_status', 'status', 'state', 'condition'], 'found_label_columns': [], 'found_status_columns': ['machine_status'], 'found_counts': {'labels_found': 0, 'status_found': 1}, 'chosen_column_stats': {'top_values': {'NORMAL': 205836, 'RECOVERING': 14477, 'BROKEN': 7}, 

{'ts_utc': '2026-03-01T04:18:26.515925+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'resolve_label_or_status_source',
 'message': 'Resolved label/status source column for Silver.',
 'why': 'Silver needs one consistent source to build anomaly_flag and state flags.',
 'consequence': 'Downstream label building uses this source consistently. we also record all detected candidates for traceability.',
 'data': {'label_source_col': 'machine_status',
  'label_source_type': 'status',
  'chosen_from': 'status_candidates',
  'label_candidates': ['anomaly_flag', 'fault_flag', 'target', 'label', 'y'],
  'status_candidates': ['machine_status', 'status', 'state', 'condition'],
  'found_label_columns': [],
  'found_status_columns': ['machine_status'],
  'found_counts': {'labels_found': 0, 'status_found': 1},
  'chosen_column_stats': {'top_values': {'NORMAL': 205836,
    'RECOVERING': 14477,
    'BROKEN': 7},
   'unique_count': 3,
   'no

In [35]:
dataframe.columns

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__label_type', 'meta__label_source', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id',
       'meta__record_id', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25',
       'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39',
       'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status', 'meta__layer',
       'meta__processed_at_utc', 'meta__silver_

### Protect the Cananoical Time/Event Column Names


In [36]:
def protect_canonical_output_names(
        dataframe: pd.DataFrame,
        *,
        canonical_output_columns: List[str],
        raw_prefix: str = "raw__",
    ) -> Tuple[pd.DataFrame, List[str], List[str], Dict[str, str]]:

    """
    If the input dataset already contains a canonical output name (event_time/event_step/time_index),
    rename the existing column to raw__<name> (or a unique variant) so we don't overwrite it.

    Returns:
      - updated dataframe
      - rename_map {original_name: new_name}
    """

    dataframe = dataframe.copy()

    rename_map: Dict[str, str] = {}

    for canonical_name in canonical_output_columns:
        if canonical_name not in dataframe.columns:
            continue

        base_new_name = f"{raw_prefix}{canonical_name}"
        new_name = base_new_name

        # This should ensure uniqueness
        counter = 2
        while new_name in dataframe.columns:
            new_name = f"{base_new_name}__{counter}"
            counter += 1

        dataframe = dataframe.rename(columns={canonical_name: new_name})
        rename_map[canonical_name] = new_name

    return dataframe, rename_map

In [37]:

# Run protection
dataframe, rename_map = protect_canonical_output_names(
    dataframe,
    canonical_output_columns=CANONICAL_OUTPUT_COLUMNS,
    raw_prefix = "raw__"
)

# json my rename map
rename_map_json = {str(key): str(value) for key, value in rename_map.items()}


#### #### #### #### #### #### #### #### 


if rename_map_json:
    ledger.add(
        kind="step",
        step="canonical_name_collision_protection",
        message="Renamed input columns that collide with canonical outputs (Policy A).",
        why="Prevent overwriting raw source columns when creating canonical outputs.",
        consequence="Original values preserved under raw__*; canonical outputs can be created safely.",
        data={
            "policy": "A",
            "canonical_outputs": list(CANONICAL_OUTPUT_COLUMNS),
            "raw_prefix": "raw__",
            "collisions": int(len(rename_map_json)),
            "renames": rename_map_json,
            "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
        },
        logger=logger,
    )
else:
    ledger.add(
        kind="step",
        step="canonical_name_collision_protection",
        message="No canonical-name collisions found (Policy A).",
        why="Confirm input does not contain columns that conflict with canonical outputs.",
        consequence="Canonical outputs can be created without renaming any source columns.",
        data={
            "policy": "A",
            "canonical_outputs": list(CANONICAL_OUTPUT_COLUMNS),
            "collisions": 0,
            "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
        },
        logger=logger,
    )

rename_map_json


2026-03-01 04:18:27,691 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:27.691082+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'canonical_name_collision_protection', 'message': 'No canonical-name collisions found (Policy A).', 'why': 'Confirm input does not contain columns that conflict with canonical outputs.', 'consequence': 'Canonical outputs can be created without renaming any source columns.', 'data': {'policy': 'A', 'canonical_outputs': ['event_time', 'event_step', 'time_index'], 'collisions': 0, 'shape': {'rows': 220320, 'cols': 74}}}


{}

In [38]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [39]:
def _pick_first_existing_candidate_column(dataframe: pd.DataFrame, candidates: List[str]) -> Optional[str]: 
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate
    return None


def _ensure_grouping_columns_exists(dataframe: pd.DataFrame, *, asset_column: str = "meta__asset_id", run_column: str = "meta__run_id", default_asset_value: str = "asset__single", default_run_value: str = "run__single") -> pd.Dataframe:
    if asset_column not in dataframe.columns:
        dataframe[asset_column] = default_asset_value
    if run_column not in dataframe.columns:
        dataframe[run_column] = default_run_value
    return dataframe


def evaluate_time_candidates(dataframe: pd.DataFrame, *, candidates: List[str], minimum_parse_success_percent: float) -> Dict[str, Any]:

    chosen_time_column = None
    chosen_parse_time_series = None
    chosen_parse_success_percent = None
    
    for candidate in candidates:
        if candidate not in dataframe.columns:
            continue

        parsed_time_series = pd.to_datetime(dataframe[candidate], errors="coerce", utc=True)

        if len(dataframe) == 0:
            parse_success_percent = 0.0
        else:
            parse_success_percent = float(parsed_time_series.notna().mean() * 100.0)

        if parse_success_percent >= minimum_parse_success_percent:
            chosen_time_column = candidate
            chosen_parse_time_series = parsed_time_series
            chosen_parse_success_percent = parse_success_percent
            break

    return {
        "time_column_used": chosen_time_column,
        "time_parse_success_percent": chosen_parse_success_percent,
        "parsed_time_series": chosen_parse_time_series,
    }


def evaluate_step_candidates(dataframe: pd.DataFrame, *, candidates: List[str], minimum_parse_success_percent: float) -> Dict[str, Any]:

    chosen_step_column = None
    chosen_parse_success_percent = None
    
    for candidate in candidates:
        if candidate not in dataframe.columns:
            continue

        numeric_series = pd.to_numeric(dataframe[candidate], errors="coerce")

        if len(dataframe) == 0:
            parse_success_percent = 0.0
        else:
            parse_success_percent = float(numeric_series.notna().mean() * 100.0)

        if parse_success_percent >= minimum_parse_success_percent:
            chosen_step_column = candidate
            chosen_parse_success_percent = parse_success_percent
            break

    return {
        "step_column_used": chosen_step_column,
        "step_parse_success_percent": chosen_parse_success_percent,
    }

def build_canonical_identity_and_order_master(dataframe: pd.DataFrame, *, dataset_name: str, time_candidates: List[str], step_candidates: List[str], tie_breaker_candidates: List[str], minimum_time_parse_success_percent: float = 95.0, minimum_step_parse_success_percent: float = 95.0, default_asset_id: str = "asset__single", default_run_id: str = "run__single") -> Tuple[pd.Dataframe, Dict[str, Any]]:
    
    dataframe_copy = dataframe.copy()

    dataframe_copy = _ensure_grouping_columns_exists(dataframe_copy, default_asset_value=default_asset_id, default_run_value=default_run_id,)

    group_columns = ["meta__asset_id", "meta__run_id"]

    tie_breaker_column_used =  _pick_first_existing_candidate_column(dataframe_copy, candidates = tie_breaker_candidates)

    time_evaluation = evaluate_time_candidates(dataframe_copy, candidates=time_candidates, minimum_parse_success_percent=minimum_time_parse_success_percent)

    step_evaluation = evaluate_step_candidates(dataframe_copy, candidates=step_candidates, minimum_parse_success_percent=minimum_step_parse_success_percent)

    time_column_used = time_evaluation["time_column_used"]
    
    time_parse_success_percent = time_evaluation["time_parse_success_percent"]

    parsed_time_series = time_evaluation["parsed_time_series"]

    step_column_used = step_evaluation["step_column_used"]
    step_parse_success_percent = step_evaluation["step_parse_success_percent"]

    if time_column_used is not None:
        ordering_mode = "time"
        dataframe_copy["event_time"] = parsed_time_series
        dataframe_copy["meta__event_time_source"] = str(time_column_used)
        dataframe_copy["meta__event_time_parse_success_percent"] = float(time_parse_success_percent)
    else:
        dataframe_copy["event_time"] = pd.NaT
        dataframe_copy["meta__event_time_source"] = ""
        dataframe_copy["meta__event_time_parse_success_percent"] = 0.0

        if step_column_used is not None:
            ordering_mode = "step"
            dataframe_copy["meta__event_step_source"] = str(step_column_used)
            dataframe_copy["meta__event_step_parse_success_percent"] = float(step_parse_success_percent)
        else:
            ordering_mode = "row"
            dataframe_copy["meta__event_step_source"] = "row_order"
            dataframe_copy["meta__event_step_parse_success_percent"] = 0.0

    sort_columns: List[str] = []
    sort_columns.extend(group_columns)

    if ordering_mode == "time":
        sort_columns.append("event_time")
    elif ordering_mode == "step":
        sort_columns.append(step_column_used)

    if tie_breaker_column_used is not None:
        sort_columns.append(tie_breaker_column_used)

    should_sort = bool(ordering_mode != "row" or tie_breaker_column_used is not None)

    if should_sort and len(sort_columns) > 0:
        dataframe_copy = dataframe_copy.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)

    dataframe_copy["event_step"] = dataframe_copy.groupby(group_columns, dropna=False).cumcount()
    dataframe_copy["time_index"] = dataframe_copy["event_step"].astype("int64")

    dataframe_copy["meta__event_id"] = (
        str(dataset_name)
        + ":" + dataframe_copy["meta__asset_id"].astype(str)
        + ":" + dataframe_copy["meta__run_id"].astype(str)
        + ":" + dataframe_copy["event_step"].astype(str)
    )

    if pd.api.types.is_datetime64_any_dtype(dataframe_copy["event_time"]):
        dataframe_copy["event_date"] = dataframe_copy["event_time"].dt.floor("D")
    else:
        dataframe_copy["event_date"] = pd.NaT

    group_count = int(dataframe_copy[group_columns].drop_duplicates().shape[0])


    info = {
        "ordering_mode": ordering_mode,
        "group_columns": group_columns,
        "group_count": group_count,
        "rows": int(len(dataframe_copy)),
        "time_column_used": time_column_used,
        "time_parse_success_percent": time_parse_success_percent,
        "step_column_used": step_column_used,
        "step_parse_success_percent": step_parse_success_percent,
        "tie_breaker_column_used": tie_breaker_column_used,
        "sorted": bool(should_sort),
        "sort_columns": sort_columns,
    }

    return dataframe_copy, info

In [40]:

dataframe, canonical_info = build_canonical_identity_and_order_master(
    dataframe,
    dataset_name=DATASET_NAME,
    time_candidates=TIME_COLUMN_CANDIDATES,
    step_candidates=STEP_COLUMN_CANDIDATES,
    tie_breaker_candidates=TIE_BREAKER_CANDIDATES,
    minimum_time_parse_success_percent=MIN_TIME_PARSE_SUCCESS_PERCENT,
    minimum_step_parse_success_percent=MIN_STEP_PARSE_SUCCESS_PERCENT,
    default_asset_id=ASSET_ID_DEFAULT_FALLBACK,
    default_run_id=RUN_ID_DEFAULT_FALLBACK,
)

#### #### #### #### #### #### #### #### 
ledger.add(
    kind="decision",
    step="build_canonical_identity_and_order_master",
    message="Built canonical identity + ordering master per (asset, run).",
    why="Silver requires a stable per-(asset,run) event axis for windowing, segmentation, EDA, and consistent downstream modeling across datasets.",
    consequence="Each (asset,run) group has event_step/time_index from 0..N-1; event_time is parsed when available; meta__event_id is deterministic.",
    data={
        "dataset_name": DATASET_NAME,
        "time_candidates": list(TIME_COLUMN_CANDIDATES),
        "step_candidates": list(STEP_COLUMN_CANDIDATES),
        "tie_breaker_candidates": list(TIE_BREAKER_CANDIDATES),
        **canonical_info,
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


# Quick preview
preview_columns = [
    "meta__asset_id", "meta__run_id",
    "event_time", "event_step", "time_index",
    "meta__event_id", "event_date",
    "meta__event_time_source", "meta__event_step_source",
]

preview_columns = [column for column in preview_columns if column in dataframe.columns]

dataframe[preview_columns].head(10), canonical_info

2026-03-01 04:18:29,487 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:29.487263+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'build_canonical_identity_and_order_master', 'message': 'Built canonical identity + ordering master per (asset, run).', 'why': 'Silver requires a stable per-(asset,run) event axis for windowing, segmentation, EDA, and consistent downstream modeling across datasets.', 'consequence': 'Each (asset,run) group has event_step/time_index from 0..N-1; event_time is parsed when available; meta__event_id is deterministic.', 'data': {'dataset_name': 'pump', 'time_candidates': ['timestamp', 'event_time', 'time', 'datetime', 'date'], 'step_candidates': ['cycle', 'step', 'event_step', 'sample', 'time_index'], 'tie_breaker_candidates': ['meta__source_row_id', 'meta__record_id'], 'ordering_mode': 'time', 'group_columns': ['meta__asset_id', 'meta__run_id'], 'group_count': 1, 'rows': 220320, 'time_col

(  meta__asset_id meta__run_id                event_time  event_step  time_index              meta__event_id                event_date meta__event_time_source
 0     asset__001     run__001 2018-04-01 00:00:00+00:00           0           0  pump:asset__001:run__001:0 2018-04-01 00:00:00+00:00               timestamp
 1     asset__001     run__001 2018-04-01 00:01:00+00:00           1           1  pump:asset__001:run__001:1 2018-04-01 00:00:00+00:00               timestamp
 2     asset__001     run__001 2018-04-01 00:02:00+00:00           2           2  pump:asset__001:run__001:2 2018-04-01 00:00:00+00:00               timestamp
 3     asset__001     run__001 2018-04-01 00:03:00+00:00           3           3  pump:asset__001:run__001:3 2018-04-01 00:00:00+00:00               timestamp
 4     asset__001     run__001 2018-04-01 00:04:00+00:00           4           4  pump:asset__001:run__001:4 2018-04-01 00:00:00+00:00               timestamp
 5     asset__001     run__001 2018-04-01 00:0

In [41]:
dataframe.columns

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__label_type', 'meta__label_source', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id',
       'meta__record_id', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25',
       'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39',
       'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status', 'meta__layer',
       'meta__processed_at_utc', 'meta__silver_

In [42]:
ANOMALY_FLAG_COLUMN = "anomaly_flag"

def normalize_label_to_binary(
    series: pd.Series,
) -> Tuple[pd.Series, Dict[str, Any]]:
    """
    Convert a label-like series to 0/1.
    Handles common patterns:
      - already numeric 0/1
      - booleans
      - strings like "normal"/"anomaly", "ok"/"fail", "true"/"false", "yes"/"no"
    Unknown/unhandled values become NaN then filled as 0 (conservative).
    """

    raw_series = series.copy()

    # Try numeric first
    numeric_series = pd.to_numeric(raw_series, errors="coerce")

    # If numeric has enough coverage, use it
    numeric_non_null_percent = float(numeric_series.notna().mean() * 100.0) if len(raw_series) else 0.0

    if numeric_non_null_percent >= 95.0:
        # Normalize: anything > 0 -> 1, else 0
        binary_series = (numeric_series.fillna(0) > 0).astype("int64")
        info = {
            "method": "numeric_threshold_gt_0",
            "numeric_non_null_percent": numeric_non_null_percent,
        }
        return binary_series, info

    # Fall back to string normalization
    string_series = raw_series.astype("string").str.strip().str.lower()

    mapping = {
        "1": 1, "true": 1, "yes": 1, "y": 1, "anomaly": 1, "fault": 1, "fail": 1, "failed": 1, "broken": 1,
        "0": 0, "false": 0, "no": 0, "n": 0, "normal": 0, "ok": 0, "good": 0, "healthy": 0, "running": 0,
    }

    mapped = string_series.map(mapping)
    mapped_non_null_percent = float(mapped.notna().mean() * 100.0) if len(mapped) else 0.0

    # Conservative fill: unknowns become 0 (normal)
    binary_series = mapped.fillna(0).astype("int64")

    info = {
        "method": "string_mapping_conservative_fill",
        "mapped_non_null_percent": mapped_non_null_percent,
        "numeric_non_null_percent": numeric_non_null_percent,
        "mapping_keys_count": int(len(mapping)),
    }
    return binary_series, info



def build_anomaly_flag_from_status(
    series: pd.Series,
    *,
    normal_value: Optional[str] = None,
) -> Tuple[pd.Series, Dict[str, Any]]:
    """
    Convert a status-like series into anomaly_flag:
      anomaly_flag = 1 if status != normal_value else 0

    If normal_value is None, choose the mode (most frequent non-null).
    """
    raw_series = series.astype("string").str.strip()

    # Remove empty strings
    cleaned = raw_series.where(raw_series != "", other=pd.NA)

    # Determine normal value
    chosen_normal_value = normal_value
    if chosen_normal_value is None:
        non_null_values = cleaned.dropna()
        if len(non_null_values) == 0:
            chosen_normal_value = None
        else:
            chosen_normal_value = str(non_null_values.value_counts().index[0])

    if chosen_normal_value is None:
        # No usable normal value; default all normal (0)
        anomaly_flag = pd.Series(np.zeros(len(series), dtype=np.int64), index=series.index)
        info = {
            "method": "status_no_non_null_values",
            "normal_value": None,
        }
        return anomaly_flag, info

    anomaly_flag = (cleaned.astype("string") != str(chosen_normal_value)).fillna(False).astype("int64")

    info = {
        "method": "status_compare_to_normal_value",
        "normal_value": str(chosen_normal_value),
        "unique_status_count": int(cleaned.nunique(dropna=True)),
        "non_null_count": int(cleaned.notna().sum()),
    }
    return anomaly_flag, info


In [43]:

anomaly_build_info: Dict[str, Any] = {
    "source_type": LABEL_SOURCE_TYPE, 
    "source_column": LABEL_SOURCE_COLUMN
    }

if LABEL_SOURCE_TYPE == "label" and LABEL_SOURCE_COLUMN:
    anomaly_series, method_info = normalize_label_to_binary(dataframe[LABEL_SOURCE_COLUMN])
    dataframe[ANOMALY_FLAG_COLUMN] = anomaly_series
    anomaly_build_info.update(method_info)

elif LABEL_SOURCE_TYPE == "status" and LABEL_SOURCE_COLUMN:
    anomaly_series, method_info = build_anomaly_flag_from_status(dataframe[LABEL_SOURCE_COLUMN], normal_value=None)
    dataframe[ANOMALY_FLAG_COLUMN] = anomaly_series
    anomaly_build_info.update(method_info)

    # Optional state flags (handy for EDA)
    normal_value_text = anomaly_build_info.get("normal_value")
    if normal_value_text is not None:
        cleaned_status = dataframe[LABEL_SOURCE_COLUMN].astype("string").str.strip()
        dataframe["status_normal_value"] = str(normal_value_text)
        dataframe["is_normal"] = (cleaned_status == str(normal_value_text)).fillna(False).astype("int64")
        dataframe["is_anomaly"] = (dataframe[ANOMALY_FLAG_COLUMN] == 1).astype("int64")

else:
    # No label/status available: default to all normal
    dataframe[ANOMALY_FLAG_COLUMN] = np.zeros(len(dataframe), dtype=np.int64)
    anomaly_build_info.update({"method": "no_label_or_status_available_default_all_normal"})


# Basic summary for ledger
anomaly_counts = dataframe[ANOMALY_FLAG_COLUMN].value_counts(dropna=False).to_dict()
anomaly_counts_json = {str(key): int(value) for key, value in anomaly_counts.items()}

anomaly_build_info["anomaly_flag_counts"] = anomaly_counts_json
anomaly_build_info["anomaly_rate_percent"] = float((dataframe[ANOMALY_FLAG_COLUMN].mean() * 100.0)) if len(dataframe) else 0.0

#### #### #### #### #### #### #### #### 
ledger.add(
    kind="step",
    step="build_anomaly_flag",
    message="Built anomaly_flag from resolved label/status source.",
    why="Silver requires a consistent binary anomaly flag for segmentation, evaluation, and synthetic anomaly generation.",
    consequence="Downstream steps can rely on anomaly_flag (0/1) being present across datasets.",
    data={
        "label_source_column": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        **anomaly_build_info,
        "shape": {"rows": int(len(dataframe)), "columns": int(len(dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

# # Quick preview
preview_columns = [column for column in [LABEL_SOURCE_COLUMN, "meta__label_source", "meta__label_type", "status_normal_value", "is_normal", "is_anomaly", "anomaly_flag"] if column and column in dataframe.columns]
dataframe[preview_columns].head(12), anomaly_build_info

2026-03-01 04:18:30,582 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:30.582372+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'step', 'step': 'build_anomaly_flag', 'message': 'Built anomaly_flag from resolved label/status source.', 'why': 'Silver requires a consistent binary anomaly flag for segmentation, evaluation, and synthetic anomaly generation.', 'consequence': 'Downstream steps can rely on anomaly_flag (0/1) being present across datasets.', 'data': {'label_source_column': 'machine_status', 'label_source_type': 'status', 'source_type': 'status', 'source_column': 'machine_status', 'method': 'status_compare_to_normal_value', 'normal_value': 'NORMAL', 'unique_status_count': 3, 'non_null_count': 220320, 'anomaly_flag_counts': {'0': 205836, '1': 14484}, 'anomaly_rate_percent': 6.5740740740740735, 'shape': {'rows': 220320, 'columns': 85}}}


(   machine_status meta__label_source meta__label_type status_normal_value  is_normal  is_anomaly  anomaly_flag
 0          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 1          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 2          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 3          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 4          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 5          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 6          NORMAL               <NA>             <NA>              NORMAL          1           0             0
 7          NORMAL               <NA>             <NA>              NORMAL          1           0       

In [44]:
dataframe.columns

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__label_type', 'meta__label_source', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id',
       'meta__record_id', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25',
       'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39',
       'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status', 'meta__layer',
       'meta__processed_at_utc', 'meta__silver_

In [45]:

DEFAULT_EXCLUDE_PREFIXES = ["meta__", "raw__"]

def classify_column_type(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    if pd.api.types.is_categorical_dtype(series):
        return "categorical"
    if pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series):
        return "text"
    return "other"


def should_exclude_by_prefix(column_name: str, exclude_prefixes: List[str]) -> bool:
    for prefix in exclude_prefixes:
        if column_name.startswith(prefix):
            return True
    return False


def looks_like_identifier_column(
    dataframe: pd.DataFrame,
    *,
    column_name: str,
    unique_ratio_threshold: float = 0.50,
) -> bool:
    """
    Heuristic: exclude obvious identifier-like columns from features.
    Examples: *id*, *uuid*, *serial*, etc. or extremely high-cardinality columns.
    """
    lower_name = column_name.lower()

    identifier_keywords = ["id", "uuid", "guid", "serial", "record"]
    keyword_hit = False
    for keyword in identifier_keywords:
        if keyword in lower_name:
            keyword_hit = True
            break

    series = dataframe[column_name]
    total_rows = int(len(series))

    if total_rows == 0:
        return False

    unique_count = int(series.nunique(dropna=True))
    unique_ratio = float(unique_count / total_rows)

    # If it looks like an ID name AND has lots of unique values, treat it like an identifier
    if keyword_hit and unique_ratio >= unique_ratio_threshold:
        return True

    # If it is extremely high-cardinality text, treat as identifier/text signal rather than numeric feature
    if (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)) and unique_ratio >= 0.80:
        return True

    return False





def identify_feature_set(
    dataframe: pd.DataFrame,
    *,
    exclude_prefixes: List[str],
    exclude_columns: List[str],
    label_source_column: Optional[str],
    include_categorical_features: bool = False,
    include_text_features: bool = False,
    include_datetime_features: bool = False,
) -> Tuple[List[str], Dict[str, List[str]], Dict[str, Any]]:
    """
    Returns:
      - selected_feature_columns: List[str]
      - feature_groups: Dict[group_name, List[str]]
      - info: Dict[str, Any] with counts and decisions
    """
    candidate_columns: List[str] = []
    for column_name in dataframe.columns:
        if should_exclude_by_prefix(column_name, exclude_prefixes):
            continue
        if column_name in exclude_columns:
            continue
        if label_source_column is not None and column_name == label_source_column:
            continue
        if looks_like_identifier_column(dataframe, column_name=column_name):
            continue
        candidate_columns.append(column_name)

    feature_groups: Dict[str, List[str]] = {
        "numeric": [],
        "boolean": [],
        "categorical": [],
        "text": [],
        "datetime": [],
        "other": [],
    }

    for column_name in candidate_columns:
        column_type = classify_column_type(dataframe[column_name])
        feature_groups[column_type].append(column_name)

    selected_feature_columns: List[str] = []
    # Default: numeric + boolean (safest across datasets)
    selected_feature_columns.extend(feature_groups["numeric"])
    selected_feature_columns.extend(feature_groups["boolean"])

    if include_categorical_features:
        selected_feature_columns.extend(feature_groups["categorical"])

    if include_text_features:
        selected_feature_columns.extend(feature_groups["text"])

    if include_datetime_features:
        selected_feature_columns.extend(feature_groups["datetime"])

    # Keep stable ordering
    selected_feature_columns = sorted([str(name) for name in selected_feature_columns])

    info: Dict[str, Any] = {
        "candidate_column_count": int(len(candidate_columns)),
        "selected_feature_count": int(len(selected_feature_columns)),
        "group_counts": {
            "numeric": int(len(feature_groups["numeric"])),
            "boolean": int(len(feature_groups["boolean"])),
            "categorical": int(len(feature_groups["categorical"])),
            "text": int(len(feature_groups["text"])),
            "datetime": int(len(feature_groups["datetime"])),
            "other": int(len(feature_groups["other"])),
        },
        "selection_policy": {
            "include_categorical_features": bool(include_categorical_features),
            "include_text_features": bool(include_text_features),
            "include_datetime_features": bool(include_datetime_features),
        },
    }

    return selected_feature_columns, feature_groups, info


In [46]:

exclude_columns_combined = []
exclude_columns_combined.extend(CANONICAL_EXCLUDE_COLUMNS)
exclude_columns_combined.extend(LABEL_EXCLUDE_COLUMNS)

FEATURE_COLUMNS, FEATURE_GROUPS, FEATURE_INFO = identify_feature_set(
    dataframe,
    exclude_prefixes=DEFAULT_EXCLUDE_PREFIXES,
    exclude_columns=exclude_columns_combined,
    label_source_column=LABEL_SOURCE_COLUMN,
    include_categorical_features=False,
    include_text_features=False,
    include_datetime_features=False,
)



In [47]:
def compute_missingness_pct(
    dataframe: pd.DataFrame,
    *,
    columns: List[str],
    sort_desc: bool = True,
) -> pd.Series:
    """
    Returns a Series indexed by column name with percent missing (0..100).
    Only includes columns that exist in the dataframe.
    """
    columns = [column for column in columns if column in dataframe.columns]
    if not columns:
        return pd.Series(dtype="float64")

    missing_pct = dataframe[columns].isna().mean().mul(100.0)
    return missing_pct.sort_values(ascending=not sort_desc)


def quarantine_features_by_missingness(
    dataframe: pd.DataFrame,
    *,
    feature_columns: List[str],
    threshold_pct: float,
    drop_all_null: bool = True,
    numeric_only: bool = True,
) -> Tuple[pd.DataFrame, List[str], List[str], pd.Series]:
    """
    Drops features whose missingness >= threshold_pct (and optionally all-null).
    Returns:
      (clean_df, kept_features, dropped_features, missing_pct_series)
    """
    dataframe = dataframe.copy()

    present = [column for column in feature_columns if column in dataframe.columns]
    if numeric_only:
        present = [column for column in present if pd.api.types.is_numeric_dtype(dataframe[column])]

    missing_pct = compute_missingness_pct(dataframe, columns=present, sort_desc=True)

    if missing_pct.empty:
        return dataframe, present, [], missing_pct

    mask = missing_pct >= threshold_pct
    if drop_all_null:
        mask = mask | (missing_pct >= 100.0)

    dropped = missing_pct[mask].index.tolist()
    kept = [column for column in feature_columns if column in dataframe.columns and column not in dropped]

    if dropped:
        dataframe = dataframe.drop(columns=dropped, errors="ignore")

    return dataframe, kept, dropped, missing_pct

In [48]:
dataframe, FEATURE_COLUMNS, dropped_features, missing_pct = quarantine_features_by_missingness(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    threshold_pct=QUARANTINE_MISSING_PCT,
    drop_all_null=True,
    numeric_only=True,
)

#### #### #### #### #### #### #### #### 

logger.info("Missingness quarantine dropped %d features: %s", len(dropped_features), dropped_features)

ledger.add(
    kind="decision",
    step="missingness_quarantine",
    message="Dropped features exceeding missingness threshold (or all-null).",
    why="High-missingness features add noise/instability to EDA + modeling.",
    consequence="Feature set shrinks; downstream stats/imputation/modeling operate on remaining features.",
    data={
        "threshold_pct": float(QUARANTINE_MISSING_PCT),
        "dropped": dropped_features,
        "remaining": int(len(FEATURE_COLUMNS)),
        "top_missing_pct": missing_pct.head(10).to_dict(),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

2026-03-01 04:18:33,140 | INFO | capstone.silver | Missingness quarantine dropped 2 features: ['sensor_15', 'sensor_50']
2026-03-01 04:18:33,142 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:33.142752+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'missingness_quarantine', 'message': 'Dropped features exceeding missingness threshold (or all-null).', 'why': 'High-missingness features add noise/instability to EDA + modeling.', 'consequence': 'Feature set shrinks; downstream stats/imputation/modeling operate on remaining features.', 'data': {'threshold_pct': 30.0, 'dropped': ['sensor_15', 'sensor_50'], 'remaining': 50, 'top_missing_pct': {'sensor_15': 100.0, 'sensor_50': 34.95688090050835, 'sensor_51': 6.982116920842412, 'sensor_00': 4.633260711692085, 'sensor_07': 2.474128540305011, 'sensor_08': 2.3179920116194626, 'sensor_06': 2.177741466957153, 'sensor_09': 2.0856027596223674, 'sensor_01': 0.1674836601307189

{'ts_utc': '2026-03-01T04:18:33.142752+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'missingness_quarantine',
 'message': 'Dropped features exceeding missingness threshold (or all-null).',
 'why': 'High-missingness features add noise/instability to EDA + modeling.',
 'consequence': 'Feature set shrinks; downstream stats/imputation/modeling operate on remaining features.',
 'data': {'threshold_pct': 30.0,
  'dropped': ['sensor_15', 'sensor_50'],
  'remaining': 50,
  'top_missing_pct': {'sensor_15': 100.0,
   'sensor_50': 34.95688090050835,
   'sensor_51': 6.982116920842412,
   'sensor_00': 4.633260711692085,
   'sensor_07': 2.474128540305011,
   'sensor_08': 2.3179920116194626,
   'sensor_06': 2.177741466957153,
   'sensor_09': 2.0856027596223674,
   'sensor_01': 0.16748366013071894,
   'sensor_30': 0.1184640522875817}}}

In [49]:
if dropped_features:
    dropped_set = set(dropped_features)

    # Filter groups to only remaining features
    FEATURE_GROUPS = {
        group: [column for column in columns if column not in dropped_set and column in FEATURE_COLUMNS]
        for group, columns in FEATURE_GROUPS.items()
    }
    # Optional: prune empty groups
    FEATURE_GROUPS = {group: columns for group, columns in FEATURE_GROUPS.items() if columns}

    # Update info counts to match final schema
    FEATURE_INFO["selected_feature_count"] = int(len(FEATURE_COLUMNS))
    if "group_counts" in FEATURE_INFO:
        FEATURE_INFO["group_counts"] = {
            group: int(len(columns)) for group, columns in FEATURE_GROUPS.items()
        }
    FEATURE_INFO["missingness_quarantine"] = {
        "threshold_pct": float(QUARANTINE_MISSING_PCT),
        "dropped_count": int(len(dropped_features)),
        "dropped": list(dropped_features),
    }

In [50]:
def build_feature_set_identifier(feature_columns: List[str]) -> str:
    """
    Deterministic identifier for a feature set based on sorted column names.
    """
    normalized = "|".join(sorted([str(name) for name in feature_columns]))
    digest = hashlib.md5(normalized.encode("utf-8")).hexdigest()
    return f"feature_set__{digest}"

In [51]:
FEATURE_SET_IDENTIFIER = build_feature_set_identifier(FEATURE_COLUMNS)

# Stamp minimal metadata (do not stamp the full list into every row)
dataframe["meta__feature_set_id"] = FEATURE_SET_IDENTIFIER
dataframe["meta__feature_count"] = int(len(FEATURE_COLUMNS))



#### #### #### #### #### #### #### #### 

ledger.add(
    kind="decision",
    step="finalize_feature_set",
    message="Finalized feature set after applying feature filters (e.g., missingness quarantine).",
    why="Feature set identifier must represent the final schema used in downstream EDA/modeling.",
    consequence="Registry + parquet are consistent; Gold can assert schema alignment.",
    data={
        "feature_set_id": FEATURE_SET_IDENTIFIER,
        "label_source_column": LABEL_SOURCE_COLUMN,
        "label_source_type": LABEL_SOURCE_TYPE,
        "exclude_prefixes": list(DEFAULT_EXCLUDE_PREFIXES),
        "exclude_columns": list(exclude_columns_combined),
        "feature_count": int(len(FEATURE_COLUMNS)),
        "selected_feature_columns": list(FEATURE_COLUMNS),
        "feature_groups": {g: list(cols) for g, cols in FEATURE_GROUPS.items()},
        # If FEATURE_INFO is large, consider logging only summaries:
        # "feature_info_keys": list(FEATURE_INFO.keys())[:50],
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)


#### #### #### #### #### #### #### #### 

# Quick preview
# FEATURE_SET_IDENTIFIER, FEATURE_INFO, FEATURE_COLUMNS[:25]

2026-03-01 04:18:34,088 | INFO | capstone.silver | LEDGER | {'ts_utc': '2026-03-01T04:18:34.088156+00:00', 'stage': 'silver', 'recipe': 'silver__clean__dataset__agnostic__v001', 'kind': 'decision', 'step': 'finalize_feature_set', 'message': 'Finalized feature set after applying feature filters (e.g., missingness quarantine).', 'why': 'Feature set identifier must represent the final schema used in downstream EDA/modeling.', 'consequence': 'Registry + parquet are consistent; Gold can assert schema alignment.', 'data': {'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40', 'label_source_column': 'machine_status', 'label_source_type': 'status', 'exclude_prefixes': ['meta__', 'raw__'], 'exclude_columns': ['event_time', 'event_step', 'time_index', 'event_date', 'meta__event_id', 'meta__event_time_source', 'meta__event_step_source', 'meta__event_time_parse_success_percent', 'meta__event_step_parse_success_percent', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value',

{'ts_utc': '2026-03-01T04:18:34.088156+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'decision',
 'step': 'finalize_feature_set',
 'message': 'Finalized feature set after applying feature filters (e.g., missingness quarantine).',
 'why': 'Feature set identifier must represent the final schema used in downstream EDA/modeling.',
 'consequence': 'Registry + parquet are consistent; Gold can assert schema alignment.',
 'data': {'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40',
  'label_source_column': 'machine_status',
  'label_source_type': 'status',
  'exclude_prefixes': ['meta__', 'raw__'],
  'exclude_columns': ['event_time',
   'event_step',
   'time_index',
   'event_date',
   'meta__event_id',
   'meta__event_time_source',
   'meta__event_step_source',
   'meta__event_time_parse_success_percent',
   'meta__event_step_parse_success_percent',
   'anomaly_flag',
   'is_anomaly',
   'is_normal',
   'status_normal_value',
   'met

In [52]:
def safe_list_columns(columns: List[str], existing_columns: List[str]) -> List[str]:
    kept: List[str] = []
    for column in columns:
        if column in existing_columns:
            kept.append(column)
    return kept


def collect_meta_columns(existing_columns: List[str]) -> List[str]:
    meta_columns: List[str] = []
    for column in existing_columns:
        if column.startswith("meta__"):
            meta_columns.append(column)
    return meta_columns


def reorder_silver_columns(
    dataframe: pd.DataFrame,
    *,
    canonical_non_meta_order: List[str],
    label_columns_order: List[str],
    feature_columns: List[str],
) -> pd.DataFrame:
    existing_columns = list(dataframe.columns)

    meta_columns = collect_meta_columns(existing_columns)
    meta_columns = sorted(meta_columns)

    canonical_columns = safe_list_columns(canonical_non_meta_order, existing_columns)
    label_columns = safe_list_columns(label_columns_order, existing_columns)

    feature_columns_present = safe_list_columns(feature_columns, existing_columns)

    # Remainder columns (preserve original order for anything not in the primary groups)
    ordered_set = set(meta_columns) | set(canonical_columns) | set(label_columns) | set(feature_columns_present)

    remainder_columns: List[str] = []
    for column in existing_columns:
        if column not in ordered_set:
            remainder_columns.append(column)

    final_order: List[str] = []
    final_order.extend(meta_columns)
    final_order.extend(canonical_columns)
    final_order.extend(label_columns)
    final_order.extend(feature_columns_present)
    final_order.extend(remainder_columns)

    return dataframe[final_order].copy()

In [53]:
dataframe.columns

Index(['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__label_type', 'meta__label_source', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id',
       'meta__record_id', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26',
       'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40',
       'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_51', 'machine_status', 'meta__layer', 'meta__processed_at_utc',
       'meta__silver_version', 'meta__cleaning_

In [54]:
dataframe = reorder_silver_columns(
    dataframe,
    canonical_non_meta_order=CANONICAL_NON_META_ORDER,
    label_columns_order=LABEL_COLUMNS_ORDER,
    feature_columns=FEATURE_COLUMNS,
)

In [55]:
dataframe.columns

Index(['meta__asset_id', 'meta__cleaning_recipe_id', 'meta__dataset', 'meta__dataset_name', 'meta__dataset_source', 'meta__event_id', 'meta__event_time_parse_success_percent',
       'meta__event_time_source', 'meta__feature_count', 'meta__feature_set_id', 'meta__has_label_candidates', 'meta__has_status_candidates', 'meta__ingested_at_utc', 'meta__label_source',
       'meta__label_source_column', 'meta__label_source_kind', 'meta__label_type', 'meta__layer', 'meta__processed_at_utc', 'meta__record_id', 'meta__run_id', 'meta__silver_version',
       'meta__source_file', 'meta__source_row_id', 'meta__split', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value',
       'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13',
       'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'se

In [56]:
def compute_quick_quality_checks(
    dataframe: pd.DataFrame,
    *,
    feature_columns: List[str],
    event_id_column: str = "meta__event_id",
    anomaly_flag_column: str = "anomaly_flag",
) -> Dict[str, Any]:
    total_rows = int(len(dataframe))

    # Duplicate checks
    duplicate_row_count = int(dataframe.duplicated().sum())

    duplicate_event_id_count = None
    if event_id_column in dataframe.columns:
        duplicate_event_id_count = int(dataframe[event_id_column].duplicated().sum())

    # Missingness for feature columns (limit logging size)
    numeric_missingness: Dict[str, float] = {}
    numeric_feature_count = 0

    for column in feature_columns:
        if column not in dataframe.columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[column]):
            numeric_feature_count += 1
            missing_percent = float(dataframe[column].isna().mean() * 100.0) if total_rows > 0 else 0.0
            numeric_missingness[column] = missing_percent

    # Keep only top 25 missingness entries (largest first) to avoid huge ledger payload
    sorted_missingness = sorted(numeric_missingness.items(), key=lambda item: item[1], reverse=True)
    top_missingness = dict(sorted_missingness[:25])

    # Anomaly rate
    anomaly_rate_percent = None
    anomaly_counts = None
    if anomaly_flag_column in dataframe.columns and total_rows > 0:
        anomaly_rate_percent = float(dataframe[anomaly_flag_column].mean() * 100.0)
        value_counts = dataframe[anomaly_flag_column].value_counts(dropna=False).to_dict()
        anomaly_counts = {str(key): int(value) for key, value in value_counts.items()}

    return {
        "total_rows": total_rows,
        "total_columns": int(len(dataframe.columns)),
        "duplicate_row_count": duplicate_row_count,
        "duplicate_event_id_count": duplicate_event_id_count,
        "numeric_feature_count": int(numeric_feature_count),
        "top_numeric_missingness_percent": top_missingness,
        "anomaly_rate_percent": anomaly_rate_percent,
        "anomaly_counts": anomaly_counts,
    }

In [57]:
quality_info = compute_quick_quality_checks(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
)

In [58]:

feature_registry: Dict[str, Any] = {
    "dataset_name": DATASET_NAME,
    "row_count": int(len(dataframe)),
    "column_count": int(len(dataframe.columns)),
    "feature_set_id": FEATURE_SET_IDENTIFIER,
    "feature_count": int(len(FEATURE_COLUMNS)),
    "feature_columns": list(FEATURE_COLUMNS),
    "feature_groups": {
        group_name: list(columns) for group_name, columns in FEATURE_GROUPS.items()
    },
    "feature_info": FEATURE_INFO,
    "exclude_prefixes": list(DEFAULT_EXCLUDE_PREFIXES),
    "exclude_columns": list(exclude_columns_combined),
    "label_source_column": LABEL_SOURCE_COLUMN,
    "label_source_type": LABEL_SOURCE_TYPE,
    "quarantine_missing_pct": float(QUARANTINE_MISSING_PCT),
}


In [59]:
if not dataframe.columns.is_unique:
    duplicates_found = dataframe.columns[dataframe.columns.duplicated()].tolist()

    raise ValueError(f"Duplicate columns detected before save: {duplicates_found}")

In [60]:
dataframe.head()

,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:0,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,14598431322315673869,run__001,silver__001,sensor.csv,0,unsplit,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:00:00,NORMAL
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:1,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,15954729095895098000,run__001,silver__001,sensor.csv,1,unsplit,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:01:00,NORMAL
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:2,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,10041703297090838359,run__001,silver__001,sensor.csv,2,unsplit,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,203.7037,2018-04-01 00:02:00,NORMAL
3,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:3,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816

In [61]:
# Save Data as Parquet

#silver_output_folder = Path.cwd() / "artifacts" / "silver"

#silver_file_name = f"silver__{DATASET_NAME}.parquet"


# Save Silver Parquet
saved_parquet_path = save_data(
    dataframe,
    file_path=SILVER_TRAIN_DATA_PATH,
    file_name=SILVER_TRAIN_DATA_FILE_NAME,
    create_dirs=True,
    index=False,
)

# Save feature registry JSON
saved_registry_path = save_json(
    feature_registry,
    file_path=SILVER_ARTIFACTS_PATH,
    file_name=f"silver__{DATASET_NAME}__feature_registry.json",
    create_dirs=True,
    indent=2,
)



#### #### #### #### #### #### #### #### 


ledger.add(
    kind="step",
    step="silver_finalize_export",
    message="Finalized Silver schema, ran quick quality checks, exported Parquet, and saved feature registry.",
    why="A stable Silver export and feature registry is the foundation for reproducible EDA, modeling, and synthetic anomaly generation.",
    consequence="Downstream steps can load the Parquet and feature registry as the single source of truth for Silver.",
    data={
        "dataset_name": DATASET_NAME,
        "silver_parquet_path": str(saved_parquet_path),
        "feature_registry_path": str(saved_registry_path),
        "quality_info": quality_info,
        "feature_set_id": FEATURE_SET_IDENTIFIER,
        "feature_count": int(len(FEATURE_COLUMNS)),
        "shape": {"rows": int(len(dataframe)), "cols": int(len(dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 





2026-03-01 04:18:38,052 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-03-01 04:18:40,158 | INFO | capstone.file_io | Saved: pump__silver__train.parquet | shape=(220320, 85) | columns=['meta__asset_id', 'meta__cleaning_recipe_id', 'meta__dataset', 'meta__dataset_name', 'meta__dataset_source', 'meta__event_id', 'meta__event_time_parse_success_percent', 'meta__event_time_source', 'meta__feature_count', 'meta__feature_set_id', 'meta__has_label_candidates', 'meta__has_status_candidates', 'meta__ingested_at_utc', 'meta__label_source', 'meta__label_source_column', 'meta__label_source_kind', 'meta__label_type', 'meta__layer', 'meta__processed_at_utc', 'meta__record_id', 'meta__run_id', 'meta__silver_version', 'meta__source_file', 'meta__source_row_id', 'meta__split', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_0

{'ts_utc': '2026-03-01T04:18:40.182874+00:00',
 'stage': 'silver',
 'recipe': 'silver__clean__dataset__agnostic__v001',
 'kind': 'step',
 'step': 'silver_finalize_export',
 'message': 'Finalized Silver schema, ran quick quality checks, exported Parquet, and saved feature registry.',
 'why': 'A stable Silver export and feature registry is the foundation for reproducible EDA, modeling, and synthetic anomaly generation.',
 'consequence': 'Downstream steps can load the Parquet and feature registry as the single source of truth for Silver.',
 'data': {'dataset_name': 'pump',
  'silver_parquet_path': '/workspace/data/silver/train/pump__silver__train.parquet',
  'feature_registry_path': '/workspace/artifacts/silver/pump/silver__pump__feature_registry.json',
  'quality_info': {'total_rows': 220320,
   'total_columns': 85,
   'duplicate_row_count': 0,
   'duplicate_event_id_count': 0,
   'numeric_feature_count': 50,
   'top_numeric_missingness_percent': {'sensor_51': 6.982116920842412,
    'sen

In [62]:
# Save the ledger
saved_ledger_path = ledger.write_json(
    SILVER_ARTIFACTS_PATH / f"silver__{DATASET_NAME}__ledger.json"
)


In [63]:
finalize_info = finalize_wandb_stage(
    run=wandb_run,
    stage=STAGE,
    dataframe=dataframe,
    project_root=paths.root,
    logs_dir=paths.logs,
    dataset_dirs=[paths.data_silver_train],
    dataset_artifact_name=f"{DATASET_NAME}-{STAGE}-dataset",
    logger=logger,
    notebook_path=None,
    aliases=("latest",),
    table_key=None,
    table_n=15,
    profile=True,
)

# Close the W&B run
wandb_run.finish()

2026-03-01 04:18:41,939 | INFO | capstone.silver | Shape: (220320, 85)
2026-03-01 04:18:41,941 | INFO | capstone.silver | Memory (MB): 374.37
2026-03-01 04:18:41,944 | INFO | capstone.silver | Dtypes:
meta__asset_id                                         object
meta__cleaning_recipe_id                               object
meta__dataset                                        category
meta__dataset_name                                     object
meta__dataset_source                                   object
meta__event_id                                         object
meta__event_time_parse_success_percent                float64
meta__event_time_source                                object
meta__feature_count                                     int64
meta__feature_set_id                                   object
meta__has_label_candidates                              int64
meta__has_status_candidates                             int64
meta__ingested_at_utc                     datetime64[us

bronze_cols,▁
bronze_rows,▁
cols,▁
memory_mb,▁
rows,▁
bronze_cols,65
bronze_rows,220320
cols,85
memory_mb,374.37201
rows,220320
